[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/autolabel_pitch.ipynb)

# Pitch auto-label proposals (Phase 3.5b active-learning loop)
Predict (pose-vN) -> fit anchors -> propagate homographies -> project canonical
landmarks back into frames as YOLO-pose pre-labels. Correct these in Roboflow.
Pre-labels inherit propagation error (a few px) — review before training on them.

In [ ]:
!pip install -q "git+https://github.com/PatrickJReed/soccer-vision.git#subdirectory=packages/soccer-vision[roboflow]"
from pathlib import Path
VIDEO = Path("/content/game.mp4")     # a training game
WEIGHTS = Path("/content/pitch_v0.pt")  # current pose model
OUT = Path("/content/autolabels"); OUT.mkdir(exist_ok=True)
MIN_CONF = 0.5MAX_GAP = 45


In [ ]:
from soccer_vision.tracking.roboflow import RoboflowBackend
backend = RoboflowBackend(detect_pitch=True, pitch_weights_path=WEIGHTS)
trajectories, keypoints = backend.process_with_pitch(VIDEO)
print("frames with keypoints:", keypoints['frame'].nunique())

In [ ]:
import cv2
from soccer_vision.pitch.landmarks import build_frame_homographies, PITCH_LANDMARKS
from soccer_vision.pitch.propagation import (
    compute_interframe_homographies, propagate_homographies,
)

anchors = build_frame_homographies(keypoints)
cap = cv2.VideoCapture(str(VIDEO))
def read_frame(i):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, f = cap.read()
    return f if ok else None

keys = sorted(anchors)
needed = {i for a, b in zip(keys, keys[1:]) if b - a - 1 <= MAX_GAP for i in range(a, b)}
interframe = compute_interframe_homographies(read_frame, needed, trajectories)
homs = propagate_homographies(anchors, interframe, max_gap=MAX_GAP)
print("homographies (anchor+propagated):", len(homs))

In [ ]:
from soccer_vision.pitch.autolabel import propose_labels, to_yolo_pose_line

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
proposals = propose_labels(homs, PITCH_LANDMARKS, (w, h), min_confidence=MIN_CONF)
img_dir = OUT / "images"; lbl_dir = OUT / "labels"
img_dir.mkdir(exist_ok=True); lbl_dir.mkdir(exist_ok=True)
for frame, kpts in proposals.items():
    img = read_frame(frame)
    if img is None:
        continue
    cv2.imwrite(str(img_dir / f"f{frame:06d}.jpg"), img)
    (lbl_dir / f"f{frame:06d}.txt").write_text(to_yolo_pose_line(kpts, (w, h)))
cap.release()
print("wrote", len(proposals), "pre-labeled frames")

In [ ]:
import matplotlib.pyplot as plt
sample = sorted(proposals)[len(proposals)//2]
img = cv2.cvtColor(cv2.imread(str(img_dir / f"f{sample:06d}.jpg")), cv2.COLOR_BGR2RGB)
for x, y, v in proposals[sample]:
    if v > 0:
        cv2.circle(img, (int(x), int(y)), 6, (255, 0, 0), -1)
plt.figure(figsize=(12, 7)); plt.imshow(img); plt.title(f"frame {sample}"); plt.axis("off")

In [ ]:
import shutil
shutil.make_archive("/content/autolabels", "zip", OUT)
from google.colab import files
files.download("/content/autolabels.zip")